# 28 — Dependency Vulnerability Scanner

Paste a requirements.txt and get a ranked CVE report in seconds. OSV.dev (no API key) supplies the raw vulnerability data. The LLM classifies severity, ranks packages, and writes the action plan.

In [ ]:
!pip install -q openai python-dotenv pydantic

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from pydantic import BaseModel, Field

class Vulnerability(BaseModel):
    vuln_id: str = Field(description='CVE or GHSA identifier.')
    severity: str = Field(description='CRITICAL, HIGH, MEDIUM, LOW, or UNKNOWN.')
    summary: str = Field(description='One-sentence description of what the vulnerability allows.')
    fixed_in: list[str] = Field(description='Versions where this is fixed.')

class PackageRisk(BaseModel):
    package: str = Field(description='Package name.')
    version: str = Field(description='Version scanned.')
    ecosystem: str = Field(description='PyPI, npm, etc.')
    vulnerabilities: list[Vulnerability] = Field(description='All CVEs for this package version.')
    highest_severity: str = Field(description='Highest severity across all CVEs: CRITICAL, HIGH, MEDIUM, LOW, or NONE.')

class VulnerabilityReport(BaseModel):
    packages_scanned: int = Field(description='Total packages checked.')
    packages_with_findings: int = Field(description='Packages with at least one CVE.')
    critical_count: int = Field(description='Total CRITICAL CVEs.')
    high_count: int = Field(description='Total HIGH CVEs.')
    findings: list[PackageRisk] = Field(description='Packages with CVEs, sorted CRITICAL first.')
    risk_summary: str = Field(description='2-3 sentences: which packages to upgrade first and why.')

In [ ]:
import json
import urllib.request
from openai import OpenAI

OSV_URL = 'https://api.osv.dev/v1/query'
SEVERITY_RANK = {'CRITICAL': 4, 'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, 'UNKNOWN': 0}

SYSTEM = (
    'You are a security analyst. Given raw CVE data from OSV.dev:\n'
    '- List all CVEs per package with severity, one-sentence summary, and fix versions\n'
    '- Assign highest_severity as the worst CVE severity for that package\n'
    '- Sort findings CRITICAL first\n'
    '- Count packages_with_findings, critical_count, high_count accurately\n'
    '- Write a 2-3 sentence risk_summary naming the most urgent upgrades\n'
    'Use only provided data. Do not invent vulnerabilities.'
)

def _query_osv(package, version, ecosystem='PyPI'):
    payload = json.dumps({'version': version, 'package': {'name': package, 'ecosystem': ecosystem}}).encode()
    req = urllib.request.Request(OSV_URL, data=payload, headers={'Content-Type': 'application/json'}, method='POST')
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read()).get('vulns', [])
    except Exception:
        return []

def _severity(vuln):
    for aff in vuln.get('affected', []):
        s = aff.get('database_specific', {}).get('severity', '').upper()
        if s in SEVERITY_RANK:
            return s
    return 'UNKNOWN'

def _fixed(vuln):
    out = []
    for aff in vuln.get('affected', []):
        for r in aff.get('ranges', []):
            for e in r.get('events', []):
                if 'fixed' in e:
                    out.append(e['fixed'])
    return list(set(out))

def parse_reqs(text):
    pkgs = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        for sep in ('==', '>=', '~='):
            if sep in line:
                name, rest = line.split(sep, 1)
                pkgs.append((name.strip(), rest.strip().split(',')[0].strip()))
                break
    return pkgs

def scan(requirements_txt):
    packages = parse_reqs(requirements_txt)
    raw = []
    for name, ver in packages:
        vulns = _query_osv(name, ver)
        if vulns:
            raw.append({'package': name, 'version': ver, 'ecosystem': 'PyPI',
                        'vulns': [{'id': v.get('id',''), 'summary': v.get('summary',''), 'severity': _severity(v), 'fixed_in': _fixed(v)} for v in vulns]})
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    r = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[{'role':'system','content':SYSTEM}, {'role':'user','content':f'Packages scanned: {len(packages)}\n\nRaw data:\n{json.dumps(raw, indent=2)}'}],
        response_format=VulnerabilityReport
    )
    report = r.choices[0].message.parsed
    report.packages_scanned = len(packages)
    return report

In [ ]:
# Legacy data science stack -- several known-vulnerable old versions
LEGACY = '''
pillow==9.0.1
pyyaml==5.3.1
cryptography==36.0.0
urllib3==1.26.5
setuptools==65.5.0
'''

report = scan(LEGACY)
print(f'Scanned: {report.packages_scanned} | With CVEs: {report.packages_with_findings} | Critical: {report.critical_count} | High: {report.high_count}')
for pkg in report.findings:
    print(f'\n[{pkg.highest_severity}] {pkg.package}=={pkg.version}')
    for v in pkg.vulnerabilities:
        fix = f' -> fix: {", ".join(v.fixed_in)}' if v.fixed_in else ''
        print(f'  {v.vuln_id} ({v.severity}): {v.summary}{fix}')
print(f'\nRisk summary: {report.risk_summary}')

In [ ]:
# Mixed stack -- mostly current, one outdated package
MIXED = '''
requests==2.31.0
pydantic==2.5.0
pillow==9.4.0
openai==1.12.0
'''

report2 = scan(MIXED)
print(f'Scanned: {report2.packages_scanned} | With CVEs: {report2.packages_with_findings}')
if report2.findings:
    for pkg in report2.findings:
        print(f'\n[{pkg.highest_severity}] {pkg.package}=={pkg.version}')
        for v in pkg.vulnerabilities:
            fix = f' -> fix: {", ".join(v.fixed_in)}' if v.fixed_in else ''
            print(f'  {v.vuln_id} ({v.severity}): {v.summary}{fix}')
    print(f'\nRisk summary: {report2.risk_summary}')
else:
    print('No vulnerabilities found.')

## Exercise

Paste your own requirements.txt into `scan()` and see what comes back. Try `django==3.0.0` or `flask==1.0.0` — both have known CVEs in those old versions. The OSV.dev data is live, so results reflect the current advisory state.